In [ ]:
!pip install -q transformers accelerate bitsandbytes mistral_inference vllm huggingface_hub

# Huggingface login

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
huggingface_api = user_secrets.get_secret("huggingface_api")
login(token=huggingface_api)

# Utils

In [ ]:
import time
import json
import torch
import re
import os
from tqdm import tqdm
import transformers
from typing import Dict, Any, List, Tuple
from datetime import datetime
import sys
from vllm import LLM, SamplingParams
import gc


## Random seed setting

In [4]:
import torch
import numpy as np
import random
import os

def setup_reproducible_environment(seed: int = 42):
    """
    Setup reproducible environment for scientific experiments.
    
    Args:
        seed: Random seed for reproducibility
        
    Note:
        This function ensures deterministic behavior across runs
        for scientific reproducibility requirements.
    """
    # Python random
    random.seed(seed)
    
    # NumPy random
    np.random.seed(seed)
    
    # PyTorch random
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU
    
    # Ensure deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set environment variable for additional determinism
    os.environ['PYTHONHASHSEED'] = str(seed)

setup_reproducible_environment()

## Prompt Technical

In [5]:
def create_zero_shot_prompt(text, sent_id) -> Tuple[str, str]:
    """
    Tạo prompt cho zero-shot sentiment analysis với định dạng đầu ra cụ thể.
    
    Args:
        text: Vietnamese text to analyze
        sent_id: Unique identifier for the sentence
        
    Returns:
        Tuple of (system_prompt, user_prompt) for model input
        
    Note:
        System prompt contains detailed instructions for Vietnamese
        structured sentiment analysis with JSON output format.
    """
    system_prompt = """Bạn là chuyên gia phân tích cảm xúc tiếng Việt có cấu trúc. Nhiệm vụ của bạn là phân tích bình luận mạng xã hội và trích xuất các thành phần cảm xúc theo cấu trúc JSON.

ĐỊNH NGHĨA CÁC THÀNH PHẦN:

1. SOURCE (Nguồn gốc bình luận):
   - Người phát biểu ý kiến, có thể là người bình luận hoặc được trích dẫn
   - Thường là các đại từ nhân xưng: "Tôi", "Tao", "Mình", "Bọn tao", "Mẹ tui"
   - Có thể có hoặc không có trong câu

2. TARGET (Đối tượng hướng tới):
   - Cá nhân, tập thể, sự vật, hiện tượng mà bình luận hướng đến
   - Thường là các đại từ xưng hô: "Mày", "Cậu", "Anh ấy", "Bạn"
   - Có thể có hoặc không có trong câu

3. POLAR_EXPRESSION (Biểu thức cảm xúc):
   - Từ/cụm từ bày tỏ cảm xúc, ý nghĩ, cảm nhận, hành động
   - Bao gồm: tính từ cảm xúc, thán từ, hành động xúc phạm/khen ngợi
   - Ví dụ: "buồn", "vui", "tức giận", "đáng đời", "đánh"
   - BẮT BUỘC phải có

4. POLARITY (Tính chất cảm xúc):
   - Positive: Khích lệ, động viên, chia sẻ, vui đùa không xúc phạm
   - Negative: Xúc phạm, kích động, chia rẽ, gây thù ghét
   - Neutral: Bình luận bình thường, khách quan

5. INTENSITY (Cường độ cảm xúc):
   - Strong: Cảm xúc mạnh mẽ, từ ngữ quyết liệt
   - Standard: Cảm xúc bình thường, từ ngữ thông thường  
   - Weak: Cảm xúc nhẹ nhàng, từ ngữ dè dặt

QUY TẮC PHÂN TÍCH:
- Mỗi câu có thể chứa nhiều opinion khác nhau
- Mỗi opinion phải có ít nhất 1 Polar_expression
- Polar_expression là thành phần bắt buộc phải có
- Chú ý các từ viết tắt, teencode, hàm ý, ẩn ý trong tiếng Việt

FORMAT JSON OUTPUT:
{
  "text": "[Bình luận gốc]",
  "opinions": [
    {
      "Source": ["text_span_1", "text_span_2"],
      "Target": ["text_span_1", "text_span_2"],
      "Polar_expression": ["text_span_1", "text_span_2"],
      "Polarity": "Positive/Negative/Neutral",
      "Intensity": "Strong/Standard/Weak"
    }
  ]
}

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG GIẢI THÍCH THÊM, kHÔNG TRÌNH BÀY QUÁ TRÌNH SUY LUẬN.
"""

    user_prompt = f"""Phân tích cảm xúc cho văn bản sau (sent_id: {sent_id}):
"{text}"

Trả về KẾT QUẢ CHÍNH XÁC theo cấu trúc JSON đã yêu cầu."""
    return system_prompt, user_prompt

## Extract information from respone

In [6]:
def extract_position(text, expression) -> str:
    """
    Trích xuất vị trí bắt đầu và kết thúc của biểu thức trong văn bản.
    
    Args:
        text: Original text
        expression: Expression to find position for
        
    Returns:
        Position string in format "start:end" (0-indexed)
        
    Example:
        extract_position("Tôi rất vui", "rất vui") -> "4:12"
    """
    start = text.find(expression)
    if start == -1:
        return "0:0"
    end = start + len(expression)
    return f"{start}:{end}"

def postprocess_response(response_text, original_text, sent_id):
    """
    Chuẩn hóa kết quả trả về từ model để phù hợp với định dạng SemEval.
    
    Args:
        response_text: Raw JSON response from model
        original_text: Original input text
        sent_id: Sentence identifier
        
    Returns:
        Formatted JSON string with validated structure and auto-extracted positions
        
    Note:
        Handles new simplified format where components are arrays of text spans.
        Automatically extracts positions for all text spans.
    """
    try:
        result = json.loads(response_text)
        result["sent_id"] = sent_id
        result["text"] = original_text
        
        if "opinions" in result and isinstance(result["opinions"], list):
            for opinion in result["opinions"]:
                if not isinstance(opinion, dict): 
                    continue
                
                # Process each component: Source, Target, Polar_expression
                for component in ["Source", "Target", "Polar_expression"]:
                    if component not in opinion:
                        opinion[component] = [[], []]
                        continue
                    
                    component_data = opinion[component]
                    
                    # Handle new simplified format: array of text spans
                    if isinstance(component_data, list):
                        # Check if already in [texts, positions] format
                        if (len(component_data) == 2 and 
                            isinstance(component_data[0], list) and 
                            isinstance(component_data[1], list)):
                            # Old format - re-extract positions anyway for accuracy
                            texts = component_data[0]
                        else:
                            # New format - just array of text spans
                            texts = component_data
                        
                        # Clean and extract positions
                        valid_texts = [text for text in texts if isinstance(text, str) and text.strip()]
                        positions = [extract_position(original_text, text) for text in valid_texts]
                        opinion[component] = [valid_texts, positions]
                        
                    elif isinstance(component_data, str) and component_data.strip():
                        # Single string
                        text = component_data.strip()
                        position = extract_position(original_text, text)
                        opinion[component] = [[text], [position]]
                    else:
                        # Empty or invalid
                        opinion[component] = [[], []]
                
                # Set proper defaults
                if "Polarity" not in opinion or opinion["Polarity"] not in ["Positive", "Negative", "Neutral"]:
                    opinion["Polarity"] = ""
                    
                if "Intensity" not in opinion or opinion["Intensity"] not in ["Strong", "Standard", "Weak"]:
                    opinion["Intensity"] = ""
        else:
            result["opinions"] = []
            
        return json.dumps(result, ensure_ascii=False, indent=2)
        
    except json.JSONDecodeError:
        default_result = {"sent_id": sent_id, "text": original_text, "opinions": []}
        return json.dumps(default_result, ensure_ascii=False, indent=2)
    
def extract_json_from_response(response: str) -> str:
    """
    Trích xuất phần JSON từ response của model với support đa dạng format.
    
    Args:
        response: Raw model response text
        
    Returns:
        Extracted JSON string or cleaned response
        
    Note:
        Handles various response formats from different models:
        - Gemma: Assistant markers
        - Qwen: Thinking tokens  
        - Mistral/DeepSeek: Standard JSON formats
        - Universal: Backticks, greedy matching, fallbacks
    """
    original_response = response
    
    # STAGE 1: Remove model-specific markers
    
    # 1.1: Handle Gemma assistant markers
    assistant_markers = ["assistant:", "assistant", "<assistant>"]
    for marker in assistant_markers:
        if marker in response:
            response = response.split(marker, 1)[1].strip()
    
    # 1.2: Handle Qwen thinking tokens
    # Remove <think>...</think> blocks
    think_block_pattern = r"<think>[\s\S]*?</think>\s*"
    match = re.match(think_block_pattern, response, re.DOTALL)
    if match:
        response = response[match.end():].strip()
    
    # Handle <|thought|> marker
    thought_marker = "<|thought|>"
    if thought_marker in response:
        parts = response.split(thought_marker)
        response = parts[-1].strip()
    
    # STAGE 2: Extract JSON with multiple strategies
    
    # 2.1: Find JSON within triple backticks (most reliable)
    match_backticks = re.search(r'```json\s*([\s\S]*?)\s*```', response, re.DOTALL)
    if match_backticks:
        json_candidate = match_backticks.group(1).strip()
        try:
            json.loads(json_candidate)
            return json_candidate
        except json.JSONDecodeError:
            pass
    
    # 2.2: Find complete JSON object with brace matching (Qwen approach)
    first_brace_index = response.find('{')
    if first_brace_index != -1:
        open_braces = 0
        for i in range(first_brace_index, len(response)):
            if response[i] == '{':
                open_braces += 1
            elif response[i] == '}':
                open_braces -= 1
                if open_braces == 0:
                    json_candidate = response[first_brace_index : i+1]
                    try:
                        json.loads(json_candidate)
                        return json_candidate.strip()
                    except json.JSONDecodeError:
                        break
    
    # 2.3: Greedy JSON matching (Mistral/DeepSeek approach)
    json_pattern_greedy = r'({[\s\S]*})'
    match_greedy = re.search(json_pattern_greedy, response)
    if match_greedy:
        json_candidate_greedy = match_greedy.group(1).strip()
        try:
            json.loads(json_candidate_greedy)
            return json_candidate_greedy
        except json.JSONDecodeError:
            pass
    
    # 2.4: Find all potential JSON objects and test (fallback)
    json_pattern_findall = r'({[\s\S]*?})'
    json_matches = re.findall(json_pattern_findall, response)
    if json_matches:
        # Try from largest to smallest (reversed order)
        for match_str in reversed(json_matches):
            candidate = match_str.strip()
            try:
                json.loads(candidate)
                return candidate
            except json.JSONDecodeError:
                continue
    
    # STAGE 3: Final fallback
    return response.strip()

## Other functions

In [7]:
def save_experiment_metadata(
    config: Dict[str, Any], 
    results_summary: Dict[str, Any], 
    output_dir: str = "/kaggle/working"
) -> str:
    """
    Save experiment metadata for reproducibility tracking.
    
    Args:
        config: Experiment configuration parameters
        results_summary: Summary of experiment results
        output_dir: Directory to save metadata
        
    Returns:
        Path to saved metadata file
        
    Example:
        config = {
            "model_id": "google/gemma-3-12b-it",
            "batch_size": 8,
            "max_tokens": 512
        }
        save_experiment_metadata(config, results)
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Get GPU info if available
    gpu_info = {}
    if torch.cuda.is_available():
        gpu_info = {
            "gpu_count": torch.cuda.device_count(),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "Unknown",
            "cuda_version": torch.version.cuda
        }
    
    metadata = {
        "timestamp": timestamp,
        "experiment_config": config,
        "results_summary": results_summary,
        "environment": {
            "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
            "torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "gpu_info": gpu_info
        }
    }
    
    # Save metadata
    metadata_file = os.path.join(output_dir, f"experiment_metadata_{timestamp}.json")
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"📊 Experiment metadata saved: {metadata_file}")
    return metadata_file

def load_dataset(dataset_path) -> List[Dict[str, Any]]:
    """Load dataset from JSON file"""
    try:
        with open(dataset_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Successfully loaded {len(data)} samples from {dataset_path}")
        return data
    except Exception as e:
        print(f"Error loading dataset from {dataset_path}: {e}")
        return []
    
def gpu_memory_cleanup():
    """Clean up GPU memory - important for T4 with limited VRAM."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        # More aggressive cleanup for T4 GPUs
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
    print("🧹 GPU memory cleanup completed")
        
def print_result(result_json, gen_time):
    """
    Hiển thị kết quả phân tích dạng JSON với định dạng đẹp.
    
    Args:
        result_json: JSON result string to display
        gen_time: Generation time in seconds
        
    Note:
        Handles both valid JSON and fallback raw output display.
    """
    try:
        result = json.loads(result_json)
        print("\n" + "="*50)
        print(f"KẾT QUẢ PHÂN TÍCH CẢM XÚC (thời gian: {gen_time:.2f}s):")
        print("-"*50)
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("="*50)
    except json.JSONDecodeError: # Changed from generic except
        print("\n" + "="*50)
        print(f"KẾT QUẢ PHÂN TÍCH CẢM XÚC (thời gian: {gen_time:.2f}s) - RAW OUTPUT (JSON PARSE FAILED):")
        print("-"*50)
        print(result_json)
        print("="*50)

# URA-LLAMA-13b

## Config

In [10]:
# ================================
# URA LLaMA CONFIGURATION WITH vLLM
# ================================

# Model Configuration
MODEL_ID = "ura-hcmut/ura-llama-13b"
TRUST_REMOTE_CODE = True

# vLLM Configuration - Optimized for Kaggle 2x T4 GPUs
TENSOR_PARALLEL_SIZE = 2  # Use both T4 GPUs
MAX_MODEL_LEN = 2100      # Maximum context length
GPU_MEMORY_UTILIZATION = 0.95  # Conservative for T4 16GB VRAM
DTYPE = "float16"         # Data type for inference

# Generation Configuration
DO_SAMPLE = True
TEMPERATURE = 0.1
MAX_NEW_TOKENS = 800
TOP_K = 10
REPETITION_PENALTY = 1.1
TOP_P = 0.95

# Processing Configuration - Optimized for T4 dual GPU setup
BATCH_SIZE = 8  # Conservative batch size for T4 memory constraints
CLEANUP_FREQUENCY = 8

# Dataset Configuration
DATASET_PATH = "/kaggle/input//dev.json"
OUTPUT_FILE = "results_ura_llama_vllm.json"

# Experiment Configuration
EXPERIMENT_NAME = "ura_llama_vllm_optimized"

# ================================
# END CONFIGURATION
# ================================


## Inference

In [11]:
### -*- coding: utf-8 -*-
"""LLM test performance with ura-hcmut/ura-llama-13b using vLLM for optimized inference"""
def setup_vllm_model() -> LLM:
    """
    Initialize the URA LLaMA 13B model using vLLM for optimized inference.
    Configured for Kaggle 2x T4 GPUs setup.
    
    Returns:
        vLLM LLM instance ready for inference
    """
    print(f"Initializing vLLM model: {MODEL_ID}")
    print(f"🔧 Hardware: 2x T4 GPUs, Tensor Parallel Size: {TENSOR_PARALLEL_SIZE}")
    print(f"💾 Memory utilization: {GPU_MEMORY_UTILIZATION}")
    
    # vLLM model initialization optimized for T4 GPUs
    llm = LLM(
        model=MODEL_ID,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        dtype=dtype,
        trust_remote_code=TRUST_REMOTE_CODE,
        # Additional optimizations for T4
        enforce_eager=False,  # Use CUDA graphs for better performance
        max_num_seqs=BATCH_SIZE,  # Limit concurrent sequences
    )
    
    print("✅ vLLM Model is ready for inference on 2x T4 GPUs.")
    return llm


def create_sampling_params() -> SamplingParams:
    """
    Create sampling parameters for vLLM generation.
    
    Returns:
        SamplingParams object with configured generation parameters
    """
    sampling_params = SamplingParams(
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        max_tokens=MAX_NEW_TOKENS
    )
    
    return sampling_params


def create_zero_shot_prompt_ura(text_to_analyze: str, sent_id: str) -> str:
    """
    Creates the full prompt string for ura-llama-13b,
    incorporating the system instructions and the user's text.
    
    Note: This function calls create_zero_shot_prompt which should be defined elsewhere
    """
    # Assuming create_zero_shot_prompt is defined elsewhere in your codebase
    system_instructions, user_request_for_text = create_zero_shot_prompt(text_to_analyze, sent_id)
    
    # URA-LLaMA specific prompt format
    formatted_prompt = f"[INST] <<SYS>>\n{system_instructions}\n<</SYS>>\n\n{user_request_for_text}\nTrả lời: [/INST]"
    return formatted_prompt


def generate_batch_predictions(
    llm: LLM, 
    sampling_params: SamplingParams,
    texts: List[str], 
    sent_ids: List[str]
) -> List[Tuple[str, float]]:
    """
    Generate predictions for a batch of texts using vLLM.
    
    Args:
        llm: vLLM LLM instance
        sampling_params: Sampling parameters
        texts: List of input texts
        sent_ids: List of sentence IDs
        
    Returns:
        List of tuples containing (json_response, generation_time)
    """
    # Create formatted prompts for the batch
    formatted_prompts = [
        create_zero_shot_prompt_ura(text, sent_id)
        for text, sent_id in zip(texts, sent_ids)
    ]
    
    start_time = time.time()
    
    try:
        # Generate responses using vLLM
        outputs = llm.generate(formatted_prompts, sampling_params)
        generation_time = time.time() - start_time
        
        results = []
        for output in outputs:
            # Extract generated text
            generated_text = output.outputs[0].text
            
            # Extract JSON from response
            json_response = extract_json_from_response(generated_text)
            
            # Calculate per-sample time (approximate)
            per_sample_time = generation_time / len(texts)
            results.append((json_response, per_sample_time))
        
        return results
    
    except Exception as e:
        print(f"Error in generate_batch_predictions: {str(e)}")
        # Return fallback results
        fallback_time = (time.time() - start_time) / len(texts)
        return [("{}", fallback_time) for _ in texts]


def evaluate_model_on_dataset_vllm(
    llm: LLM,
    sampling_params: SamplingParams,
    dataset: List[Dict[str, Any]], 
    output_file: str, 
    batch_size: int = BATCH_SIZE
) -> List[Dict[str, Any]]:
    """
    Evaluate URA LLaMA model on the entire dataset using vLLM and save results.
    
    Args:
        llm: vLLM LLM instance
        sampling_params: Sampling parameters
        dataset: List of data samples
        output_file: Output file path
        batch_size: Batch size for processing
        
    Returns:
        List of result dictionaries
    """
    results = []
    total_time = 0
    processed_samples_count = 0
    
    print(f"Processing {len(dataset)} samples with vLLM (batch size: {batch_size})...")
    
    for i in tqdm(range(0, len(dataset), batch_size)):
        batch_samples = dataset[i:i+batch_size]
        if not batch_samples:
            continue
        
        batch_original_texts = [sample.get("text", "") for sample in batch_samples]
        batch_sent_ids = [
            sample.get("sent_id", f"unknown_id_{i+idx}") 
            for idx, sample in enumerate(batch_samples)
        ]
        
        try:
            # Generate predictions for the batch
            batch_start_time = time.time()
            batch_results = generate_batch_predictions(
                llm, sampling_params, batch_original_texts, batch_sent_ids
            )
            batch_time = time.time() - batch_start_time
            total_time += batch_time
            
            # Process results
            for idx, (json_response_str, gen_time) in enumerate(batch_results):
                original_text = batch_original_texts[idx]
                sent_id = batch_sent_ids[idx]
                
                # Post-process response
                result_json_processed = postprocess_response(
                    json_response_str, original_text, sent_id
                )
                
                try:
                    result_dict = json.loads(result_json_processed)
                    results.append(result_dict)
                except json.JSONDecodeError:
                    print(f"Failed to parse JSON for sent_id {sent_id}")
                    results.append({
                        "sent_id": sent_id,
                        "text": original_text,
                        "opinions": [],
                        "error": "json_parse_error_after_postprocess"
                    })
                
                processed_samples_count += 1
        
        except Exception as e:
            print(f"Error processing batch starting with ID {batch_sent_ids[0] if batch_sent_ids else 'N/A'}: {str(e)}")
            # Add error results for the batch
            for idx_err in range(len(batch_samples)):
                results.append({
                    "sent_id": batch_sent_ids[idx_err] if idx_err < len(batch_sent_ids) else f"error_unknown_id_{i+idx_err}",
                    "text": batch_original_texts[idx_err] if idx_err < len(batch_original_texts) else "Unknown text due to error",
                    "opinions": [],
                    "error": str(e)
                })
                processed_samples_count += 1
        
        # Cleanup after every N batches
        if (i // batch_size + 1) % CLEANUP_FREQUENCY == 0:
            gpu_memory_cleanup()
    
    # Final cleanup
    gpu_memory_cleanup()
    
    # Save results to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    avg_time = total_time / processed_samples_count if processed_samples_count > 0 else 0
    print(f"vLLM Evaluation complete. Results saved to {output_file}")
    print(f"Total time: {total_time:.2f}s, Processed samples: {processed_samples_count}, Average time per sample: {avg_time:.2f}s")
    
    return results

def main() -> None:
    """
    Main execution function - Auto dataset evaluation for URA LLaMA with vLLM.
    
    Automatically runs inference on dataset with vLLM optimization.
    """
    print("🚀 Starting URA LLaMA automatic dataset evaluation with vLLM...")
    print(f"📁 Dataset: {DATASET_PATH}")
    print(f"💾 Output: {OUTPUT_FILE}")
    print(f"📦 Batch size: {BATCH_SIZE} (optimized for T4)")
    print(f"🔧 Tensor parallel size: {TENSOR_PARALLEL_SIZE} (2x T4 GPUs)")
    print(f"💾 GPU memory utilization: {GPU_MEMORY_UTILIZATION}")
    print(f"🎯 Expected performance: 3-8x faster than transformers pipeline on T4")
    
    # Initialize vLLM model
    print("\n🔧 Initializing vLLM model...")
    llm = setup_vllm_model()
    sampling_params = create_sampling_params()
    print("✅ vLLM Model ready!")
    
    # Prepare experiment config
    experiment_config = {
        "model_id": MODEL_ID,
        "dataset_path": DATASET_PATH,
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_k": TOP_K,
        "top_p": TOP_P,
        "repetition_penalty": REPETITION_PENALTY,
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "max_model_len": MAX_MODEL_LEN,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "dtype": dtype,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "experiment_name": EXPERIMENT_NAME
    }
    
    # Load dataset
    print(f"\n📥 Loading dataset from {DATASET_PATH}...")
    dataset = load_dataset(DATASET_PATH)
    if not dataset:
        print("❌ Dataset empty or not found. Exiting.")
        return
    
    print(f"✅ Loaded {len(dataset)} samples")
    
    # Run evaluation
    print(f"\n🏃 Starting vLLM evaluation...")
    start_time = time.time()
    results = evaluate_model_on_dataset_vllm(
        llm, sampling_params, dataset, OUTPUT_FILE, batch_size=BATCH_SIZE
    )
    total_experiment_time = time.time() - start_time
    
    # Prepare results summary
    results_summary = {
        "total_samples": len(dataset),
        "processed_samples": len(results),
        "total_time_seconds": total_experiment_time,
        "avg_time_per_sample": total_experiment_time / len(results) if results else 0,
        "output_file": OUTPUT_FILE,
        "success_rate": len([r for r in results if "error" not in r]) / len(results) if results else 0,
        "speedup_estimate": "3-8x faster than transformers pipeline on 2x T4 GPUs"
    }
    
    # Save experiment metadata
    print("\n💾 Saving experiment metadata...")
    save_experiment_metadata(experiment_config, results_summary)
    
    # Final summary
    print("\n" + "="*70)
    print("🎉 URA LLaMA vLLM EVALUATION COMPLETED!")
    print("="*70)
    print(f"📊 Processed: {results_summary['processed_samples']}/{results_summary['total_samples']} samples")
    print(f"⏱️  Total time: {results_summary['total_time_seconds']:.2f}s")
    print(f"🚀 Avg time/sample: {results_summary['avg_time_per_sample']:.2f}s")
    print(f"✅ Success rate: {results_summary['success_rate']:.2%}")
    print(f"💾 Results saved: {OUTPUT_FILE}")
    print(f"🔥 Expected speedup: {results_summary['speedup_estimate']}")
    print("💡 Tip: Monitor GPU memory usage with 'nvidia-smi' during inference")
    print("="*70)

In [ ]:
if __name__ == "__main__":
    main()

## Tester

In [ ]:
class ModelTester:
    """
    Optimized testing framework for multiple inference tests without model reload for URA LLaMA using vLLM.
    Shows detailed comparison between raw responses and processed results.
    """
    
    def __init__(self):
        self.llm = None
        self.sampling_params = None
        self.is_initialized = False
        
    def initialize_model(self):
        """Initialize vLLM model and sampling parameters once for multiple tests."""
        if not self.is_initialized:
            print("🔧 Initializing URA LLaMA vLLM model...")
            self.llm = setup_vllm_model() # Use setup_vllm_model
            self.sampling_params = create_sampling_params() # Use create_sampling_params
            self.is_initialized = True
            print("✅ vLLM Model ready for testing!")
        else:
            print("✅ vLLM Model already initialized!")
    
    def test_single_sample(self, text: str, sent_id: str, show_details: bool = True):
        """
        Test inference on single sample with detailed output comparison using vLLM.
        
        Args:
            text: Input text to analyze
            sent_id: Sample identifier
            show_details: Whether to show detailed breakdown
            
        Returns:
            Dict with all processing stages
        """
        if not self.is_initialized:
            self.initialize_model()
            
        print(f"\n🧪 TESTING SAMPLE: {sent_id}")
        print("="*60)
        print(f"📝 Input text: '{text}'")
        
        # STAGE 1: Generate raw response using vLLM
        print(f"\n🚀 Stage 1: Generating raw response...")
        try:
            # vLLM generate for single prompt, returns a list of outputs
            formatted_prompt = create_zero_shot_prompt(text, sent_id)
            
            start_time = time.time()
            outputs = self.llm.generate([formatted_prompt], self.sampling_params)
            gen_time = time.time() - start_time
            
            if outputs and outputs[0].outputs:
                raw_response_text = outputs[0].outputs[0].text
                json_response_str = extract_json_from_response(raw_response_text)
            else:
                print(f"Warning: Unexpected output format from vLLM: {outputs}")
                json_response_str = "{}"
            
            print(f"✅ Generation successful! Time: {gen_time:.2f}s")
            
        except Exception as e:
            print(f"❌ Generation failed: {str(e)}")
            return {"error": str(e)}
        
        # STAGE 2: Extract JSON (already handled by generate_prediction in this setup)
        json_extracted = json_response_str 
        print(f"✅ JSON extraction complete!")
        
        # STAGE 3: Postprocess (add positions, validate format)
        print(f"\n🛠️  Stage 3: Postprocessing...")
        final_result = postprocess_response(json_extracted, text, sent_id)
        print(f"✅ Postprocessing complete!")
        
        # DISPLAY RESULTS
        if show_details:
            self._display_detailed_results(raw_response_text, json_extracted, final_result, gen_time)
            
        return {
            "raw_response": raw_response_text,
            "json_extracted": json_extracted,
            "final_result": final_result,
            "generation_time": gen_time,
            "success": True
        }
    
    def _display_detailed_results(self, raw_response, json_extracted, final_result, gen_time):
        """Display detailed comparison of processing stages."""
        print(f"\n📊 DETAILED RESULTS COMPARISON:")
        print("="*80)
        
        print(f"\n🔸 STAGE 1 - RAW MODEL RESPONSE:")
        print("-" * 40)
        print(f"Length: {len(raw_response)} characters")
        print(f"Content preview: {raw_response[:200]}...")
        if len(raw_response) > 200:
            print(f"... (+{len(raw_response)-200} more characters)")
            
        print(f"\n🔸 STAGE 2 - EXTRACTED JSON:")
        print("-" * 40)
        print(f"Length: {len(json_extracted)} characters")
        try:
            import json
            parsed = json.loads(json_extracted)
            print(f"Valid JSON: ✅ {len(parsed.get('opinions', []))} opinions found")
        except:
            print("Valid JSON: ❌ Invalid JSON format")
        print(f"Content: {json_extracted}")
        
        print(f"\n🔸 STAGE 3 - FINAL PROCESSED RESULT:")
        print("-" * 40)
        try:
            import json
            final_parsed = json.loads(final_result)
            print(f"✅ Final JSON valid")
            print(f"📍 Sent ID: {final_parsed.get('sent_id', 'N/A')}")
            print(f"📄 Text: {final_parsed.get('text', 'N/A')}")
            print(f"💭 Opinions: {len(final_parsed.get('opinions', []))}")
            
            for i, opinion in enumerate(final_parsed.get('opinions', [])):
                print(f"   Opinion {i+1}:")
                print(f"     Source: {opinion.get('Source', [[], []])}")
                print(f"     Target: {opinion.get('Target', [[], []])}")
                print(f"     Expression: {opinion.get('Polar_expression', [[], []])}")
                print(f"     Polarity: {opinion.get('Polarity', 'N/A')}")
                print(f"     Intensity: {opinion.get('Intensity', 'N/A')}")
        except Exception as e:
            print(f"❌ Final JSON invalid: {str(e)}")
            
        print(f"\n⏱️  PERFORMANCE: {gen_time:.2f}s generation time")
        print("="*80)
    
    def test_multiple_samples(self, samples: list):
        """Test multiple samples efficiently using vLLM."""
        if not self.is_initialized:
            self.initialize_model()
            
        print(f"\n🧪 TESTING {len(samples)} SAMPLES")
        print("="*60)
        
        results = []
        # Reusing the batch prediction logic from evaluate_model_on_dataset_vllm for consistency
        # This will allow ModelTester to leverage batching even for a list of samples.
        
        # Prepare lists for batching
        batch_texts = [sample[0] for sample in samples]
        batch_sent_ids = [sample[1] for sample in samples]

        try:
            # Generate predictions for the batch using vLLM's batching capability
            batch_results_from_vllm = generate_batch_predictions(
                self.llm, self.sampling_params, batch_texts, batch_sent_ids
            )
            
            for idx, (json_response_str, gen_time_per_sample) in enumerate(batch_results_from_vllm):
                original_text = batch_texts[idx]
                sent_id = batch_sent_ids[idx]
                
                # Post-process response
                final_result = postprocess_response(json_response_str, original_text, sent_id)
                
                # Append result, similar to test_single_sample's return structure
                results.append({
                    "raw_response": json_response_str, # In vLLM context, this is already the extracted JSON
                    "json_extracted": json_response_str,
                    "final_result": final_result,
                    "generation_time": gen_time_per_sample,
                    "success": True # Assume success unless parsing fails
                })
                print(f"\n📋 Sample {idx+1}/{len(samples)} processed.")
                if self.is_initialized: # Only show details if ModelTester initialized
                    self._display_detailed_results(json_response_str, json_response_str, final_result, gen_time_per_sample)

        except Exception as e:
            print(f"Error processing batch in ModelTester: {str(e)}")
            for idx_err in range(len(samples)): 
                results.append({
                    "sent_id": samples[idx_err][1], # Access sent_id from original samples
                    "text": samples[idx_err][0],    # Access text from original samples
                    "opinions": [],
                    "error": str(e)
                })

        # Summary
        successful = sum(1 for r in results if r.get('success', False) and "error" not in r)
        print(f"\n📊 BATCH TEST SUMMARY:")
        print(f"✅ Successful: {successful}/{len(samples)}")
        print(f"❌ Failed: {len(samples)-successful}/{len(samples)}")
        
        return results
    
    def cleanup(self):
        """Clean up GPU memory. (vLLM handles its own memory, explicit cleanup might not be needed as much)"""
        if self.is_initialized:
            # vLLM manages its own memory efficiently. Explicit cleanup might be less critical.
            # However, if there are other torch tensors outside vLLM, this helps.
            gpu_memory_cleanup() 
            print("🧹 GPU memory cleanup completed (if applicable for non-vLLM tensors).")
            
TEST_SAMPLES = [
    ("Bữa nay ít chửi, nói chuyện nhẹ nhàng hơn, livechim duyên dáng nc vs fan vui tính, ăn mặc sexy vs chất hơn 👌👌 Ủng hộ chị 🔥", "test_001"),
    ("Có mấy thằng cộng nô lên mạng đọc báo lề phải cũng tin răm rắp", "test_002"),
    ("Mai thầy nhớ lập kênh mới ở đâu để bọn em điểm danh nhé", "test_003"),
    ("làm chết người không phân biệt được cố ý và vô ý à, chưa kể tình tiết thế nào m biết không mà phán thế ?", "test_004")
]
# We will not call ModelTester directly in the main execution block
tester = ModelTester()
result = tester.test_multiple_samples(TEST_SAMPLES)